# Inference Speed in Action  - Part II

In this notebook, you'll compare a model running on a GPU against one running on Cerebras. You'll send the same prompt to both backends, measure their latency metrics, then repeat the comparison across different prompts to see how the speed gap holds up. 

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook, 2) click on <em>"Open"</em>, and then 3) click on L3.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI models can vary with each execution due to their probabilistic nature. Your results might differ from those shown in the video.</p>

## Setup

You'll compare `gpt-oss-120b`, running on Cerebras' wafer-scale engine (WSE), against the OpenAI model `gpt-4.1-mini`, running on a GPU.

**Note** that a fairer comparison would use the same model on both (`gpt-oss-120b` on WSE vs. `gpt-oss-120b` on GPU). But since OpenAI doesn't offer an endpoint for `gpt-oss-120b`, we use `gpt-4.1-mini` instead — a *smaller* model than `gpt-oss-120b`, yet you'll still see Cerebras come out faster. For reference, you can find the benchmarking results from Artificial Analysis for `gpt-oss-120b` [here](https://artificialanalysis.ai/models/gpt-oss-120b/providers).

In [ ]:
import os
import time

from cerebras.cloud.sdk import Cerebras
from dotenv import load_dotenv
from IPython.display import HTML, display
from openai import OpenAI
from rich import print

load_dotenv()

CEREBRAS_API_KEY = os.environ["CEREBRAS_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
CEREBRAS_MODEL = "gpt-oss-120b"
OPENAI_MODEL = "gpt-4.1-mini"

cerebras_client = Cerebras(api_key=CEREBRAS_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

## Side-by-side comparison

Here's a single prompt that you'll send to both providers.

In [ ]:
prompt = """
Explain in 5 short bullets why fast inference changes 
product design for LLM applications."""

Start with **Cerebras**. You call `chat.completions.create` with the model, the message, and a token budget. Since the OpenAI response won't expose timing fields, you measure it yourself with `time.perf_counter()` and compute tokens per second. This total time includes network latency, so it runs higher than the time reported inside the Cerebras response itself.

In [ ]:
cerebras_start = time.perf_counter()
cerebras_response = cerebras_client.chat.completions.create(
    model=CEREBRAS_MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=220,
)
total_time = time.perf_counter() - cerebras_start
tokens_per_second = round(cerebras_response.usage.completion_tokens / total_time, 1)

print(cerebras_response)
print(f"Total Time: {total_time:.3f}s")
print(f"Tokens per second: {tokens_per_second:.3f}s")

Now make the same call for **OpenAI**, using the same token limit so the responses are a similar length. 

In [ ]:
openai_start = time.perf_counter()
openai_response = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=220,
)
total_time = time.perf_counter() - openai_start
tokens_per_second = round(openai_response.usage.completion_tokens / total_time, 1)

print(openai_response)
print(f"Total Time: {total_time:.3f}s")
print(f"Tokens per second: {tokens_per_second:.3f}s")

## Streaming for computing the 3 latency metrics

By streaming, you can capture **time to first token (TTFT)** in addition to total time. The function `stream_chat` defined below works for both the Cerebras and OpenAI SDKs (their streaming chunk shapes match) and returns:

- `ttft`: seconds until the first content token arrives
- `total_time`: seconds until the stream finishes
- `tokens_per_second`: generated tokens per second

It also returns the fully assembled `text` so you can use the response afterwards.

In [ ]:
from pydantic import BaseModel

class StreamMetrics(BaseModel):
    ttft: float
    total_time: float
    tokens_per_second: float

def stream_chat(client, model, prompt, max_tokens=220):
    """Stream a chat completion and return (text, StreamMetrics)."""
    # The Cerebras and OpenAI SDKs use different names for the token budget.
    if isinstance(client, Cerebras):
        kwargs = {"max_completion_tokens": max_tokens}
    else:
        kwargs = {"max_tokens": max_tokens}

    start = time.perf_counter()
    ttft = None
    tokens = 0
    chunks = []

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        **kwargs,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta.content if chunk.choices else None
        if delta:
            if ttft is None:
                ttft = time.perf_counter() - start  # first token arrived
            tokens += 1
            chunks.append(delta)

    total = time.perf_counter() - start  # stream finished
    metrics = StreamMetrics(
        ttft=round(ttft, 3) if ttft is not None else 0.0,
        total_time=round(total, 3),
        tokens_per_second=round(tokens / total, 1) if total else 0.0,
    )
    return "".join(chunks), metrics

## Compare on a new prompt

Use `stream_chat` to compare all three latency metrics again, this time on a different question.

In [ ]:
new_prompt = "Summarize the history of Wikipedia in 500 words"
cerebras_text, cerebras_stream_metrics = stream_chat(cerebras_client, CEREBRAS_MODEL, new_prompt)
openai_text, openai_stream_metrics = stream_chat(openai_client, OPENAI_MODEL, new_prompt)

print("Cerebras:", cerebras_stream_metrics)
print("OpenAI:  ", openai_stream_metrics)

## Compare on the game prompt

For something more fun, you can ask each model to generate a complete 2-player pool game in a single HTML file. Same streaming pattern as before, but with a much larger token budget (`max_tokens=4000`) so there's room for the full artifact.

In [ ]:
pool_game_prompt = (
    "Create a complete, working 2-player pool game in a single self-contained HTML file. "
    "Use one fixed-size canvas that is 480 wide by 540 tall (portrait, 8:9 aspect ratio) "
    "with the table's long axis vertical and pockets along the tall sides." 
    "Set body margin to 0 so the whole game fits with no scrolling."
    "The player aims by moving the mouse and shoots by clicking. "
    "Return only the HTML."
)

In [ ]:
cerebras_game_html, cerebras_game_metrics = stream_chat(
    cerebras_client, CEREBRAS_MODEL, pool_game_prompt, max_tokens=4000
)

print("Cerebras:", cerebras_game_metrics)

In [ ]:
openai_game_html, openai_game_metrics = stream_chat(
    openai_client, OPENAI_MODEL, pool_game_prompt, max_tokens=4000
)

print("OpenAI:  ", openai_game_metrics)

## For fun: display the game Cerebras generated

Since models sometimes wrap their output in a ```` ``` ```` code fence, remove it if present and write the result to `pool_game_stream.html`.

In [ ]:
html_content = cerebras_game_html or ""
if "```" in html_content:
    parts = html_content.split("```")
    html_content = parts[1]
    if html_content.lstrip().lower().startswith("html"):
        html_content = html_content.lstrip()[4:]
html_content = html_content.strip()

with open("pool_game_stream.html", "w", encoding="utf-8") as f:
    f.write(html_content)

Finally, display the saved HTML inline with an `IFrame` so you can see (and play) the generated pool game right in the notebook.

**Note:** you may not get a fully working game on the first try, since we didn't iterate with the model. You can always run the generation again to get another one.

In [ ]:
from IPython.display import IFrame
IFrame(src="pool_game_stream.html", width=480, height=540)

## Key takeaways

- In this notebook, you called both providers the exact same way, keeping the prompt and token budget consistent so the timing comparison stays fair.
- Streaming lets you measure all three latency metrics (**TTFT**, **total time**, and **tokens per second**) not just total time.
- Across every prompt, Cerebras returned the full response faster, even though its model is *larger* than the GPU-hosted one. The speed comes from the underlying hardware.
- That speed is what makes interactive, real-time LLM experiences (like generating and rendering a playable game on the fly) practical.

## References

**Cerebras**
- [Cerebras Inference SDK (Python)](https://github.com/Cerebras/cerebras-cloud-sdk-python): the `cerebras-cloud-sdk` package used in this notebook (`from cerebras.cloud.sdk import Cerebras`).
- [Cerebras Inference documentation](https://inference-docs.cerebras.ai/): API reference, supported models, and quickstarts.
- [Cerebras inference examples](https://github.com/Cerebras/inference-examples): end-to-end sample apps.


**Benchmarks & models**
- [Artificial Analysis - gpt-oss-120b providers](https://artificialanalysis.ai/models/gpt-oss-120b/providers): independent speed/latency benchmarks across providers.
